# Mahalakshmi: GRU4Rec Dataset Experiments

Use the team's shared data loader and original-style HR/NDCG evaluator. Start with **smoke** mode, then switch to **full**. Smoke results only show that the pipeline runs.

Before running: import this notebook into Kaggle; attach the private input containing `cmpe257_kaggle_code.zip` and `dataset.pkl`; enable Internet for dependencies and a GPU for full training. This notebook uses one GPU. No GitHub token or push is needed.

Read `docs/KAGGLE_TRAINING.md` in the repo for the full walkthrough. The notebook launches Python modules from the attached source ZIP; the notebook alone is not enough.

Select `DATASET` in section 1. Each dataset uses its own weights and outputs. Start a fresh kernel when changing the dataset or code bundle. Multiple attached `dataset.pkl` files are resolved by their recorded checksum.


## 1. Load the shared code and dataset paths


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, zipfile

# Attach the PRIVATE Kaggle dataset containing these two files.
# Leave as None to discover a single matching file; set explicit paths if needed.
CODE_ZIP = None  # e.g. '/kaggle/input/cmpe257-training/cmpe257_kaggle_code.zip'
CODE_DIR = None  # Optional extracted folder containing bundle_manifest.json
DATASET = 'baby'  # 'baby', 'beauty', or 'ml-100k'
DATA_FILE = None  # e.g. '/kaggle/input/cmpe257-training/dataset.pkl'
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

def attached_file(name, override):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Expected one {name}; found {matches}. Set its path above.')
    return matches[0]

REPO = WORK_ROOT / 'CMPE257_ML'
REPO.mkdir(parents=True, exist_ok=True)
zip_matches = sorted(INPUT_ROOT.rglob('cmpe257_kaggle_code.zip'))
archive = None
if CODE_DIR is None and (CODE_ZIP is not None or zip_matches):
    archive = attached_file('cmpe257_kaggle_code.zip', CODE_ZIP)
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            if not (REPO / member).resolve().is_relative_to(REPO.resolve()):
                raise ValueError('Unsafe archive member')
        z.extractall(REPO)
else:
    # Kaggle may unpack the uploaded ZIP before attaching it to a notebook.
    manifest_path = (Path(CODE_DIR) / 'bundle_manifest.json' if CODE_DIR
                     else attached_file('bundle_manifest.json', None))
    source_dir = manifest_path.parent.resolve()
    manifest = json.loads(manifest_path.read_text())
    for name in [*manifest['files'], 'bundle_manifest.json']:
        source, target = (source_dir / name).resolve(), (REPO / name).resolve()
        if not source.is_relative_to(source_dir) or not target.is_relative_to(REPO.resolve()):
            raise ValueError('Unsafe manifest path')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)
manifest = json.loads((REPO / 'bundle_manifest.json').read_text())
for name, expected in manifest['files'].items():
    file = (REPO / name).resolve()
    if not file.is_relative_to(REPO.resolve()):
        raise ValueError('Unsafe manifest path')
    assert hashlib.sha256(file.read_bytes()).hexdigest() == expected, name
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Source branch:', manifest['branch'])
print('Base commit:', manifest['base_commit'])
if archive:
    print('Source ZIP SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
else:
    print('Loaded Kaggle-extracted source; all manifest hashes verified.')
from src.data.catalog import find_dataset, dataset_spec
DATASET_PATH = find_dataset(INPUT_ROOT, DATASET, DATA_FILE)
ARCHIVE_ROOT = WORK_ROOT if DATASET == 'baby' else WORK_ROOT / 'exports' / DATASET
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET, DATASET_PATH)


## 2. Install dependencies
If Kaggle requests a session restart after installation, restart and rerun from cell 1. Do not change package versions independently without recording the change.


In [ ]:
import subprocess
# Keep Kaggle's CUDA-enabled torch if it satisfies the shared requirements.
# Internet must be enabled for missing dependencies. Run before importing models.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--upgrade-strategy', 'only-if-needed',
                '-r', str(REPO / 'requirements-shared.txt')], check=True)


In [ ]:
import torch
from src.data import load_dataset, SequenceDataset
from src.data.load import dataset_summary
from src.training.kaggle import make_config, train, load_for_inference, predict_scores
from src.utils.reproducibility import seed_everything

print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    torch.set_num_threads(1)
    print('CPU is sufficient for smoke mode. Enable a GPU before full training.')
# Import every required model now, so dependency errors happen before training.
from sasrec import SASRec
from model import Att_Diffuse_model


## 3. Choose a run
Keep `RUN_MODE = "smoke"` for the first test. For a real experiment change it to `"full"`, choose a new `RUN_NAME`, and keep the agreed shared settings. `FINALIZE = False` saves progress without test evaluation when splitting training across sessions.


In [ ]:
RUN_MODE = 'full'  # 'full' uses all users in the selected dataset, hidden size 128, up to 500 epochs
RUN_NAME = 'mahalakshmi_gru4rec_' + RUN_MODE + '_01'
FINALIZE = True
RESUME_CHECKPOINT = None  # Path to this model's latest.pt from a saved previous run
cfg = make_config('gru4rec', RUN_MODE, dataset=DATASET)
# Example only, after agreeing with the team: cfg['train_batch_size'] = 128
# Keep evaluation_batch_size=512 for full runs, even if training batch size changes.
RUNS_ROOT = WORK_ROOT / 'runs'
if DATASET != 'baby':
    RUNS_ROOT /= DATASET
RUN_DIR = RUNS_ROOT / RUN_NAME
print(json.dumps(cfg, indent=2))
data = load_dataset(DATASET_PATH, cfg['dataset_sha256'])
print(dataset_summary(data))


## 4. Train GRU4Rec
This uses our RecBole v1.2.0 CE adapter. It trains the final target from each training history; it is not verified as the exact GRU4Rec implementation/settings used in the ADRec paper.

Validation selects the checkpoint. Test evaluation runs only at the end when `FINALIZE = True`.


In [ ]:
result = train(cfg, DATASET_PATH, RUN_DIR / 'gru4rec',
               resume=RESUME_CHECKPOINT, finalize=FINALIZE)
MODEL_DIR = RUN_DIR / 'gru4rec'


## Inspect the results
HR and NDCG are percentages at 5, 10, and 20. The shared evaluator includes the original candidate columns and averages batch means like upstream. Do not compare smoke results with full results. Test results should be used for final reporting, not tuning settings.


In [ ]:
print(json.dumps(result, indent=2))
print('Saved files:', [p.name for p in MODEL_DIR.iterdir()])


## Try inference from the saved best model
This example uses only a user's visible validation history. Item IDs are shown because this dataset has no names/images. This demo hides padding ID 0; the benchmark evaluator keeps its upstream candidate rules. Previously seen items remain eligible.


In [ ]:
model, saved_cfg = load_for_inference(MODEL_DIR / 'best.pt')
example = SequenceDataset(data, 'val', saved_cfg['max_len'])[0]
seed_everything(saved_cfg['seed'])
scores = predict_scores(model, [example['input_ids']]).cpu()
scores[:, 0] = -torch.inf  # Demo only: do not recommend padding
recommended_ids = torch.topk(scores, k=10, dim=-1).indices[0].tolist()
print('Visible history:', [i for i in example['input_ids'] if i])
print('Recommended item IDs:', recommended_ids)


## Save your work and resume later
Download `best.pt` for inference and `latest.pt` for resuming, plus `config.json`, `environment.json`, `requirements-lock.txt`, `history.json`, and `results.json`. Keep weights outside Git; share small result/config files with the team.

Use Kaggle **Save Version** to preserve notebook outputs. **Save & Run All** reruns cells from the top in a fresh session; it is a new training execution. Download or save the output before ending a session—an in-memory checkpoint cannot survive a lost session.

To resume: attach the saved output, use the identical configuration and run mode, set `RESUME_CHECKPOINT` to that model's `latest.pt`, and run into a new writable output directory. You can increase `cfg['epochs']`; other training settings must match. `FINALIZE = False` avoids testing intermediate chunks. An early-stopped run stays stopped. Resume is at the last completed epoch, not the middle of an epoch. A changed GPU/library environment may change numerical results.

If training runs out of GPU memory, start a new run with a smaller **training** batch size and record it. Keep the agreed evaluation batch size unchanged. See the guide for limitations.


## Train the remaining seeds

The first run uses seed 42. This cell trains seeds 43–46 using the same dataset and run mode. Run it once after the first model finishes. Resume an interrupted seed through section 3 using its seed, run name, and checkpoint path.


In [ ]:
for s in [43, 44, 45, 46]:
    cfg = make_config('gru4rec', RUN_MODE, dataset=DATASET)
    cfg['seed'] = s
    train(cfg, DATASET_PATH,
          RUNS_ROOT / f'mahalakshmi_gru4rec_{RUN_MODE}_seed{s}' / 'gru4rec',
          finalize=True)

In [ ]:
for folder in sorted(RUNS_ROOT.glob('*/gru4rec')):
    report = json.loads((folder / 'results.json').read_text())
    metrics = (report.get('test') or {}).get('metrics', {})
    print(folder.parent.name, '|', DATASET, '|', report['mode'],
          '| HR@10', metrics.get('HR@10'), '| NDCG@10', metrics.get('NDCG@10'))


In [ ]:
import csv, statistics

# Only export this dataset/mode. Validate all runs before copying any files.
RESULTS_DIR = WORK_ROOT / ('results' if RUN_MODE == 'full' else 'smoke_results')
if DATASET != 'baby':
    RESULTS_DIR /= DATASET
RESULTS_DIR /= 'gru4rec'
if DATASET == 'baby':
    RESULTS_DIR = WORK_ROOT / ('gru4rec_results' if RUN_MODE == 'full' else 'gru4rec_smoke_results')
records = []
seen = set()
reference = None
for folder in sorted(RUNS_ROOT.glob('*/gru4rec')):
    config = json.loads((folder / 'config.json').read_text())
    if config['mode'] != RUN_MODE:
        continue
    seed = config['seed']
    identity = {key: value for key, value in config.items() if key != 'seed'}
    if (config.get('dataset', 'baby') != DATASET
            or config['dataset_sha256'] != dataset_spec(DATASET)['sha256']
            or config['model'] != 'gru4rec'
            or reference is not None and identity != reference or seed in seen):
        raise ValueError(f'Conflicting configuration or duplicate seed: {folder}')
    report = json.loads((folder / 'results.json').read_text())
    if not report.get('test'):
        raise ValueError(f'Test evaluation is unfinished: {folder}')
    if report['test']['protocol'] != 'adrec_upstream' or report['test']['units'] != 'percent':
        raise ValueError(f'Unexpected evaluation protocol: {folder}')
    reference = identity
    seen.add(seed)
    existing = RESULTS_DIR / f'gru4rec_seed{seed}_results.json'
    if existing.exists() and json.loads(existing.read_text()) != report:
        raise ValueError(f'Conflicting exported result: {existing}')
    records.append((seed, folder, report))
if not records:
    raise ValueError('No completed runs for this dataset/mode')
records.sort()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for seed, folder, report in records:
    for name in ['config.json', 'results.json', 'history.json', 'environment.json', 'requirements-lock.txt']:
        shutil.copyfile(folder / name, RESULTS_DIR / f'gru4rec_seed{seed}_{name}')
    hashes = {name: hashlib.sha256((folder / name).read_bytes()).hexdigest()
              for name in ['best.pt', 'latest.pt']}
    (RESULTS_DIR / f'gru4rec_seed{seed}_checkpoints.json').write_text(json.dumps(
        {'dataset': DATASET, 'seed': seed, 'location': str(folder), 'files': hashes}, indent=2))
    shutil.make_archive(str(ARCHIVE_ROOT / folder.parent.name), 'zip', root_dir=folder.parent)
for name in ['environment.json', 'requirements-lock.txt']:
    shutil.copyfile(records[0][1] / name, RESULTS_DIR / name)
metrics = ['HR@5', 'HR@10', 'HR@20', 'NDCG@5', 'NDCG@10', 'NDCG@20']
rows = [dict(run_id=folder.parent.name, model='GRU4Rec', dataset=dataset_spec(DATASET)['label'],
             seed=seed, best_epoch=report['best_epoch'], epochs_completed=report['epochs_completed'],
             **{key: report['test']['metrics'][key] for key in metrics})
        for seed, folder, report in records]
seeds = [seed for seed, _, _ in records]
seed_label = f'{seeds[0]}-{seeds[-1]}' if len(seeds)>1 and seeds==list(range(seeds[0],seeds[-1]+1)) else ','.join(map(str,seeds))
common = dict(model='GRU4Rec', dataset=dataset_spec(DATASET)['label'], n_runs=len(rows), seeds=seed_label)
summary = [dict(common, statistic='mean', **{key: round(statistics.mean(r[key] for r in rows),4) for key in metrics})]
if len(rows)>1:
    summary.append(dict(common, statistic='std (sample, n-1)', **{key: round(statistics.stdev(r[key] for r in rows),4) for key in metrics}))
for filename, values in [(f'gru4rec_{DATASET}_runs.csv',rows), (f'gru4rec_{DATASET}_summary.csv',summary)]:
    with (RESULTS_DIR / filename).open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(values[0]))
        writer.writeheader(); writer.writerows(values)
print('Results:', RESULTS_DIR)
print('Archive:', shutil.make_archive(str(ARCHIVE_ROOT / ('gru4rec_results' if RUN_MODE == 'full' else 'gru4rec_smoke_results')), 'zip', root_dir=RESULTS_DIR))
print('Model ZIPs:', ARCHIVE_ROOT)


## Output locations

New dataset results are stored under `results/<dataset>/gru4rec/`, runs under `runs/<dataset>/`, and downloadable ZIPs under `exports/<dataset>/`. Baby retains the original run paths. Keep model ZIPs in `checkpoints/<dataset>/` locally, outside Git. All five seeds now export their weights as well as metrics.
